# **Semantic Layer**

## **Setup**

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

@dataclass
class SemanticConfig:
    processed_dir: Path = ROOT / "data" / "processed_reports"
    tables_dir: Path = ROOT / "data" / "processed_reports" / "tables"
    chunks_path: Path = ROOT / "data" / "processed_reports" / "chunks" / "text_chunks.parquet"
    manifest_path: Path = ROOT / "data" / "processed_reports" / "page_or_sheet_manifest.parquet"
    structured_path: Path = ROOT / "data" / "processed_reports" / "tables" / "structured_long.parquet"
    metric_dictionary_path: Path = ROOT / "data" / "processed_reports" / "tables" / "metric_dictionary.parquet"
    semantic_metrics_path: Path = ROOT / "data" / "processed_reports" / "tables" / "semantic_metrics.parquet"
    vector_records_path: Path = ROOT / "data" / "processed_reports" / "semantic_vector_records.parquet"

CFG = SemanticConfig()
CFG.tables_dir.mkdir(parents=True, exist_ok=True)
print(f"Project root: {ROOT}")

## **Artifact IO**

In [ ]:
def read_dataframe(path: Path) -> pd.DataFrame:
    if path.exists():
        if path.suffix == ".parquet":
            return pd.read_parquet(path)
        if path.suffix == ".csv":
            return pd.read_csv(path)

    csv_fallback = path.with_suffix(".csv")
    if csv_fallback.exists():
        return pd.read_csv(csv_fallback)

    print(f"Missing artifact: {path}")
    return pd.DataFrame()


def write_dataframe(df: pd.DataFrame, parquet_path: Path) -> Path:
    parquet_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(parquet_path, index=False)
        return parquet_path
    except Exception as exc:
        csv_path = parquet_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        print(f"Could not write parquet ({type(exc).__name__}: {exc}); wrote CSV instead: {csv_path}")
        return csv_path


def stable_id(*parts: Any) -> str:
    raw = "::".join("" if part is None else str(part) for part in parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()


def clean_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


chunks_df = read_dataframe(CFG.chunks_path)
manifest_df = read_dataframe(CFG.manifest_path)
structured_df = read_dataframe(CFG.structured_path)

print("chunks:", chunks_df.shape)
print("manifest:", manifest_df.shape)
print("structured:", structured_df.shape)

## **Metric Dictionary**

In [ ]:
def normalize_label(value: Any) -> str:
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9% ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


METRIC_DEFINITIONS = [
    {"canonical_metric": "revenue", "display_name": "Revenue", "category": "performance", "statement": "income_statement", "synonyms": ["revenue", "revenues", "total revenue", "net revenue", "net sales", "sales revenue", "sales"]},
    {"canonical_metric": "product_revenue", "display_name": "Product Revenue", "category": "performance", "statement": "income_statement", "synonyms": ["product revenue", "products revenue", "product sales"]},
    {"canonical_metric": "service_revenue", "display_name": "Service Revenue", "category": "performance", "statement": "income_statement", "synonyms": ["service revenue", "services revenue", "service and other revenue", "cloud revenue"]},
    {"canonical_metric": "gross_margin", "display_name": "Gross Margin", "category": "profitability", "statement": "income_statement", "synonyms": ["gross margin", "gross profit"]},
    {"canonical_metric": "operating_income", "display_name": "Operating Income", "category": "profitability", "statement": "income_statement", "synonyms": ["operating income", "income from operations", "operating profit"]},
    {"canonical_metric": "net_income", "display_name": "Net Income", "category": "profitability", "statement": "income_statement", "synonyms": ["net income", "net earnings", "net profit", "profit for the year", "earnings"]},
    {"canonical_metric": "diluted_eps", "display_name": "Diluted EPS", "category": "per_share", "statement": "income_statement", "synonyms": ["diluted eps", "diluted earnings per share", "earnings per share diluted"]},
    {"canonical_metric": "operating_cash_flow", "display_name": "Operating Cash Flow", "category": "cash_flow", "statement": "cash_flow_statement", "synonyms": ["operating cash flow", "net cash from operations", "net cash provided by operating activities"]},
    {"canonical_metric": "capital_expenditures", "display_name": "Capital Expenditures", "category": "cash_flow", "statement": "cash_flow_statement", "synonyms": ["capital expenditures", "capex", "additions to property and equipment", "property and equipment additions"]},
    {"canonical_metric": "cash_and_equivalents", "display_name": "Cash and Equivalents", "category": "balance_sheet", "statement": "balance_sheet", "synonyms": ["cash and equivalents", "cash and cash equivalents", "cash, cash equivalents, and short-term investments"]},
    {"canonical_metric": "total_assets", "display_name": "Total Assets", "category": "balance_sheet", "statement": "balance_sheet", "synonyms": ["total assets", "assets"]},
    {"canonical_metric": "total_liabilities", "display_name": "Total Liabilities", "category": "balance_sheet", "statement": "balance_sheet", "synonyms": ["total liabilities", "liabilities"]},
]


def build_metric_dictionary() -> pd.DataFrame:
    rows = []
    for definition in METRIC_DEFINITIONS:
        for synonym in definition["synonyms"]:
            rows.append({
                "canonical_metric": definition["canonical_metric"],
                "display_name": definition["display_name"],
                "category": definition["category"],
                "statement": definition["statement"],
                "synonym": synonym,
                "synonym_norm": normalize_label(synonym),
            })
    return pd.DataFrame(rows)


metric_dictionary_df = build_metric_dictionary()
metric_dictionary_df.head(10)

## **Semantic Metrics**

In [ ]:
def infer_ticker(row: pd.Series) -> str | None:
    source_text = " ".join(clean_text(row.get(col)) for col in ["source_file", "document_title", "company"])
    source_lower = source_text.lower()
    if "microsoft" in source_lower or "msft" in source_lower:
        return "MSFT"
    match = re.search(r"([A-Z]{1,6})[_-]", clean_text(row.get("source_file")))
    return match.group(1) if match else None


def infer_entity(row: pd.Series) -> str | None:
    company = clean_text(row.get("company"))
    if company:
        return company
    source_text = " ".join(clean_text(row.get(col)) for col in ["source_file", "document_title"])
    if re.search(r"microsoft|msft", source_text, flags=re.I):
        return "Microsoft"
    return None


def infer_period_fields(row: pd.Series) -> dict[str, Any]:
    context = " ".join(clean_text(row.get(col)) for col in ["period", "fiscal_year", "fiscal_quarter", "source_file", "document_title", "metric"])
    year = clean_text(row.get("fiscal_year"))
    quarter = clean_text(row.get("fiscal_quarter"))

    if not year:
        match = re.search(r"FY\s?(20\d{2}|\d{2})|(20\d{2})", context, flags=re.I)
        if match:
            year = next(group for group in match.groups() if group)
            if len(year) == 2:
                year = f"20{year}"

    if not quarter:
        match = re.search(r"Q([1-4])", context, flags=re.I)
        if match:
            quarter = f"Q{match.group(1)}"

    period_type = "fiscal_quarter" if quarter else "fiscal_year" if year else None
    period_label = " ".join(part for part in [f"FY{year}" if year else "", quarter] if part) or clean_text(row.get("period"))
    period_sort = int(year) * 10 + int(quarter[-1]) if year and quarter else int(year) * 10 if year else None

    return {
        "fiscal_year_semantic": int(year) if year and year.isdigit() else None,
        "fiscal_quarter_semantic": quarter or None,
        "period_type": period_type,
        "period_label": period_label or None,
        "period_sort": period_sort,
    }


def canonicalize_metric(row: pd.Series, dictionary: pd.DataFrame) -> dict[str, Any]:
    fields = ["metric", "row_label", "table_name", "table_context", "retrieval_text"]
    haystack = normalize_label(" ".join(clean_text(row.get(field)) for field in fields))

    best = None
    best_score = 0
    for item in dictionary.to_dict("records"):
        synonym = item["synonym_norm"]
        if not synonym:
            continue
        score = 0
        if haystack == synonym:
            score = 100
        elif re.search(rf"{re.escape(synonym)}", haystack):
            score = 90 + min(len(synonym), 9)
        elif synonym in haystack:
            score = 70 + min(len(synonym), 9)

        if score > best_score:
            best = item
            best_score = score

    if best is None:
        raw = normalize_label(row.get("metric")) or normalize_label(row.get("row_label"))
        fallback = raw.replace(" ", "_")[:80] if raw else None
        return {
            "metric_canonical": fallback,
            "metric_display_name": clean_text(row.get("metric")) or clean_text(row.get("row_label")) or None,
            "metric_category": "unmapped",
            "statement": None,
            "semantic_match_score": 0,
        }

    return {
        "metric_canonical": best["canonical_metric"],
        "metric_display_name": best["display_name"],
        "metric_category": best["category"],
        "statement": best["statement"],
        "semantic_match_score": best_score,
    }


def build_semantic_metrics(structured: pd.DataFrame, dictionary: pd.DataFrame) -> pd.DataFrame:
    if structured.empty:
        return pd.DataFrame()

    rows = []
    for _, row in structured.iterrows():
        metric_fields = canonicalize_metric(row, dictionary)
        period_fields = infer_period_fields(row)
        value = pd.to_numeric(row.get("value"), errors="coerce")

        semantic_row = {
            "semantic_id": stable_id(row.get("source_file"), row.get("sheet_name"), row.get("page_num"), row.get("table_idx"), row.get("row_idx"), row.get("metric"), row.get("raw_value")),
            "source_file": row.get("source_file"),
            "source_type": row.get("source_type"),
            "document_title": row.get("document_title"),
            "entity": infer_entity(row),
            "ticker": infer_ticker(row),
            "metric_raw": row.get("metric"),
            "row_label": row.get("row_label"),
            **metric_fields,
            "value": float(value) if pd.notna(value) else None,
            "raw_value": row.get("raw_value"),
            "units": row.get("units"),
            "currency": row.get("currency"),
            "scale": row.get("scale"),
            **period_fields,
            "sheet_name": row.get("sheet_name"),
            "page_num": row.get("page_num"),
            "table_idx": row.get("table_idx"),
            "table_name": row.get("table_name"),
            "table_context": row.get("table_context"),
            "row_idx": row.get("row_idx"),
        }
        semantic_row["retrieval_text"] = " | ".join(
            f"{key}: {clean_text(semantic_row.get(key))}"
            for key in ["entity", "ticker", "period_label", "metric_display_name", "value", "units", "currency", "source_file", "table_context"]
            if clean_text(semantic_row.get(key))
        )
        rows.append(semantic_row)

    return pd.DataFrame(rows)


semantic_metrics_df = build_semantic_metrics(structured_df, metric_dictionary_df)
print("semantic_metrics:", semantic_metrics_df.shape)
semantic_metrics_df.head(10)

## **Vector-Ready Semantic Records**

In [ ]:
VECTOR_METADATA_FIELDS = [
    "source_file", "source_type", "document_title", "entity", "company", "ticker",
    "period", "period_label", "period_type", "fiscal_year", "fiscal_year_semantic",
    "fiscal_quarter", "fiscal_quarter_semantic", "page_num", "sheet_name", "table_name",
    "metric_canonical", "metric_display_name", "metric_category", "statement",
]


def flat_metadata(row: dict[str, Any], fields: list[str]) -> dict[str, Any]:
    meta = {}
    for field in fields:
        value = row.get(field)
        if value is None or (isinstance(value, float) and pd.isna(value)):
            continue
        if isinstance(value, (str, int, float, bool)):
            meta[field] = value
        else:
            meta[field] = json.dumps(value, default=str)
    return meta


def build_text_vector_records(chunks: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    if chunks.empty:
        return records

    for _, row in chunks.iterrows():
        row_dict = row.to_dict()
        text = clean_text(row_dict.get("retrieval_text")) or clean_text(row_dict.get("text"))
        if not text:
            continue

        vector_id = stable_id("text", row_dict.get("source_file"), row_dict.get("page_num"), row_dict.get("chunk_idx"), text[:120])
        metadata = flat_metadata(row_dict, VECTOR_METADATA_FIELDS + ["chunk_idx", "section_heading", "content_type"])
        metadata["content_kind"] = "text_chunk"
        records.append({
            "vector_id": vector_id,
            "content_kind": "text_chunk",
            "document": text,
            "metadata_json": json.dumps(metadata, default=str),
        })
    return records


def build_metric_vector_records(metrics: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    if metrics.empty:
        return records

    for _, row in metrics.iterrows():
        row_dict = row.to_dict()
        text = clean_text(row_dict.get("retrieval_text"))
        if not text:
            text = " | ".join(
                f"{field}: {clean_text(row_dict.get(field))}"
                for field in ["entity", "ticker", "period_label", "metric_display_name", "value", "units", "source_file"]
                if clean_text(row_dict.get(field))
            )
        if not text:
            continue

        vector_id = stable_id("metric", row_dict.get("semantic_id"), text[:120])
        metadata = flat_metadata(row_dict, VECTOR_METADATA_FIELDS + ["semantic_id", "value", "units", "currency"])
        metadata["content_kind"] = "structured_metric"
        records.append({
            "vector_id": vector_id,
            "content_kind": "structured_metric",
            "document": text,
            "metadata_json": json.dumps(metadata, default=str),
        })
    return records


vector_records = build_text_vector_records(chunks_df) + build_metric_vector_records(semantic_metrics_df)
vector_records_df = pd.DataFrame(vector_records)
if not vector_records_df.empty:
    vector_records_df = vector_records_df.drop_duplicates(subset=["vector_id"])

print("semantic vector records:", vector_records_df.shape)
vector_records_df.head(10)

## **Persist Semantic Artifacts**

In [ ]:
written_paths = {
    "metric_dictionary": write_dataframe(metric_dictionary_df, CFG.metric_dictionary_path),
    "semantic_metrics": write_dataframe(semantic_metrics_df, CFG.semantic_metrics_path),
    "semantic_vector_records": write_dataframe(vector_records_df, CFG.vector_records_path),
}

written_paths